# 09. RQ2 rolling-origin recursive backtesting



## 0. Setup

In [1]:
import ast
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 2026
ALR_EPSILON = 1e-6
YEAR_REF = 8
BACKTEST_ORIGINS = [4, 5, 6, 7]
SMALL_N_THRESHOLD = 30
SAVE_DETAILED_ACTIVITY_PREDICTIONS = False

AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")
DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed")
OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs")
BACKTEST_DIR = OUTPUT_DIR / 'recursive_backtest'
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

print('Backtest origins:', BACKTEST_ORIGINS)
print('Outputs:', BACKTEST_DIR)

Backtest origins: [4, 5, 6, 7]
Outputs: C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs\recursive_backtest


## 1. Shared transformations, models, and metrics

In [2]:
def shares_to_alr(shares):
    clipped = np.clip(np.asarray(shares, dtype=float), ALR_EPSILON, 1)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return np.log(clipped[:, :2] / clipped[:, [2]])


def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


def add_lag_features(frame, panel_keys, value_cols, feature_level, covid_years=(5, 6)):
    prepared = frame.sort_values(panel_keys + ['year']).reset_index(drop=True).copy()
    grouped = prepared.groupby(panel_keys, sort=False)
    for column in value_cols:
        prepared[f'{column}_lag1'] = grouped[column].shift(1)
        if feature_level == 'overall':
            prepared[f'{column}_lag2'] = grouped[column].shift(2)
        else:
            prepared[f'{column}_roll2'] = grouped[column].transform(lambda s: s.shift(1).rolling(2, min_periods=1).mean())
    prepared['time_trend'] = prepared['year'] / YEAR_REF
    prepared['is_covid_year'] = prepared['year'].isin(covid_years).astype(int)
    return prepared


def add_interaction_terms(frame, group_col, expected_cols=None):
    result = frame.copy()
    dummies = pd.get_dummies(result[group_col], prefix=f'{group_col}_x_time')
    interaction = dummies.mul(result['time_trend'], axis=0)
    if expected_cols is None:
        expected_cols = list(interaction.columns)
    interaction = interaction.reindex(columns=expected_cols, fill_value=0)
    result[expected_cols] = interaction
    return result, expected_cols


def parameter_candidates(model_name):
    if model_name == 'Ridge Regression':
        return [{'alpha': .1}, {'alpha': 1.0}, {'alpha': 10.0}, {'alpha': 100.0}]
    if model_name == 'Random Forest':
        return [{'n_estimators': 160, 'max_depth': 6, 'min_samples_leaf': 5, 'max_features': .5}, {'n_estimators': 160, 'max_depth': 10, 'min_samples_leaf': 8, 'max_features': .5}, {'n_estimators': 220, 'max_depth': 8, 'min_samples_leaf': 12, 'max_features': .8}]
    return [{'n_estimators': 120, 'learning_rate': .05, 'max_depth': 2, 'min_samples_leaf': 15}, {'n_estimators': 160, 'learning_rate': .03, 'max_depth': 2, 'min_samples_leaf': 20}]


def build_model(model_name, parameters, numeric_features, categorical_features, multi_output):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if model_name == 'Ridge Regression':
        numeric_steps.append(('scale', StandardScaler()))
    preprocess = ColumnTransformer([('numeric', Pipeline(numeric_steps), numeric_features), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)])
    if model_name == 'Ridge Regression':
        estimator = Ridge(**parameters)
    elif model_name == 'Random Forest':
        estimator = RandomForestRegressor(**parameters, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        base = GradientBoostingRegressor(**parameters, random_state=RANDOM_STATE, loss='huber')
        estimator = MultiOutputRegressor(base) if multi_output else base
    return Pipeline([('preprocess', preprocess), ('model', estimator)])


def safe_mean(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if len(values) else np.nan


def safe_r2(actual, predicted):
    valid = np.isfinite(actual) & np.isfinite(predicted)
    return r2_score(actual[valid], predicted[valid]) if valid.sum() > 1 else np.nan

In [3]:
def score_prediction_frame(frame, config):
    targets = config['target_cols']
    row = {'observations': len(frame)}
    if config['is_composition']:
        actual = frame[[f'actual_{c}' for c in targets]].to_numpy(dtype=float)
        predicted = frame[[f'predicted_{c}' for c in targets]].to_numpy(dtype=float)
        absolute = np.abs(actual - predicted)
        row_error = .5 * absolute.sum(axis=1)
        row.update({'total_variation': safe_mean(row_error), 'overall_mae': safe_mean(absolute.ravel()), 'overall_rmse': float(np.sqrt(safe_mean(((actual - predicted) ** 2).ravel())))})
        for index, target in enumerate(targets):
            row[f'{target}_mae'] = safe_mean(absolute[:, index])
            row[f'{target}_rmse'] = float(np.sqrt(safe_mean((actual[:, index] - predicted[:, index]) ** 2)))
            row[f'{target}_r2'] = safe_r2(actual[:, index], predicted[:, index])
        focus_actual = actual[:, -1]
        focus_error = absolute[:, -1]
    else:
        target = targets[0]
        actual = frame[f'actual_{target}'].to_numpy(dtype=float)
        predicted = frame[f'predicted_{target}'].to_numpy(dtype=float)
        row_error = np.abs(actual - predicted)
        row.update({f'{target}_mae': safe_mean(row_error), f'{target}_rmse': float(np.sqrt(safe_mean((actual - predicted) ** 2))), f'{target}_r2': safe_r2(actual, predicted)})
        focus_actual = actual
        focus_error = row_error
    positive = focus_actual > 0
    zero = focus_actual == 0
    row['positive_cell_observations'] = int(positive.sum())
    row['positive_cell_mae'] = safe_mean(focus_error[positive])
    row['zero_cell_observations'] = int(zero.sum())
    row['zero_cell_mae'] = safe_mean(focus_error[zero])
    if 'sample_n' in frame.columns:
        sample_n = pd.to_numeric(frame['sample_n'], errors='coerce').to_numpy(dtype=float)
        small = np.isfinite(sample_n) & (sample_n < SMALL_N_THRESHOLD)
        large = np.isfinite(sample_n) & (sample_n >= SMALL_N_THRESHOLD)
        row['small_n_observations'] = int(small.sum())
        row['small_n_rate'] = float(small.sum() / np.isfinite(sample_n).sum()) if np.isfinite(sample_n).sum() else np.nan
        row['small_n_mean_error'] = safe_mean(row_error[small])
        row['large_n_observations'] = int(large.sum())
        row['large_n_mean_error'] = safe_mean(row_error[large])
    return row


def validation_score(actual, predicted, is_composition):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if is_composition:
        return float(np.mean(np.abs(actual - predicted)))
    return float(mean_absolute_error(actual.ravel(), predicted.ravel()))

## 2. Nested model selection and recursive forecasting

In [4]:
def prepare_task(frame, config):
    prepared = add_lag_features(frame, config['panel_keys'], config['target_cols'], config['feature_level'])
    prepared, interaction_cols = add_interaction_terms(prepared, config['panel_keys'][1])
    lag1_cols = [f'{c}_lag1' for c in config['target_cols']]
    extra_cols = [f'{c}_lag2' if config['feature_level'] == 'overall' else f'{c}_roll2' for c in config['target_cols']]
    numeric = lag1_cols + extra_cols + ['time_trend', 'is_covid_year']
    return prepared, lag1_cols, extra_cols, numeric, interaction_cols


def fit_one_model(model_name, parameters, train, config, numeric, interaction_cols):
    numeric_for_model = numeric + interaction_cols if model_name == 'Ridge Regression' else numeric
    categorical = config['panel_keys']
    model = build_model(model_name, parameters, numeric_for_model, categorical, config['is_composition'])
    targets = config['target_cols']
    y = shares_to_alr(train[targets]) if config['is_composition'] else train[targets[0]]
    model.fit(train[numeric_for_model + categorical], y, model__sample_weight=train[config['weight_col']].to_numpy())
    return model, numeric_for_model


def predict_one_model(model, frame, config, numeric_for_model):
    if config['is_composition']:
        return alr_to_shares(model.predict(frame[numeric_for_model + config['panel_keys']]))
    return np.clip(model.predict(frame[numeric_for_model + config['panel_keys']]), 0, 1).reshape(-1, 1)


def select_and_refit_at_origin(prepared, config, origin, lag1_cols, extra_cols, numeric, interaction_cols):
    targets = config['target_cols']
    required_features = lag1_cols + extra_cols
    train = prepared[prepared['year'] < origin].dropna(subset=targets + [config['weight_col']]).copy()
    train = train[pd.to_numeric(train[config['weight_col']], errors='coerce') > 0].copy()
    validation = prepared[prepared['year'] == origin].dropna(subset=targets + required_features).copy()
    refit = prepared[prepared['year'] <= origin].dropna(subset=targets + [config['weight_col']]).copy()
    refit = refit[pd.to_numeric(refit[config['weight_col']], errors='coerce') > 0].copy()
    if train.empty or validation.empty or refit.empty:
        raise ValueError(f"{config['task']}, origin {origin}: empty training, validation, or refit data")
    actual = validation[targets].to_numpy(dtype=float)
    naive_prediction = validation[lag1_cols].to_numpy(dtype=float)
    candidate_rows = [{'task': config['task'], 'origin': origin, 'model': 'Naive baseline', 'parameters': '{}', 'validation_score': validation_score(actual, naive_prediction, config['is_composition'])}]
    fitted_candidates = {}
    for model_name in ['Ridge Regression', 'Random Forest', 'Gradient Boosting']:
        best_score = np.inf
        best_params = None
        for parameters in parameter_candidates(model_name):
            model, numeric_for_model = fit_one_model(model_name, parameters, train, config, numeric, interaction_cols)
            predicted = predict_one_model(model, validation, config, numeric_for_model)
            score = validation_score(actual, predicted, config['is_composition'])
            if score < best_score:
                best_score = score
                best_params = parameters
        candidate_rows.append({'task': config['task'], 'origin': origin, 'model': model_name, 'parameters': json.dumps(best_params, sort_keys=True), 'validation_score': best_score})
        fitted_candidates[model_name] = best_params
    selected = min(candidate_rows, key=lambda x: x['validation_score'])
    for row in candidate_rows:
        row['selected'] = row['model'] == selected['model']
    if selected['model'] == 'Naive baseline':
        final_model = None
        numeric_for_model = numeric
    else:
        final_model, numeric_for_model = fit_one_model(selected['model'], fitted_candidates[selected['model']], refit, config, numeric, interaction_cols)
    return final_model, selected['model'], numeric_for_model, pd.DataFrame(candidate_rows)

In [5]:
def build_origin_history(frame, config, origin):
    keys = config['panel_keys']
    targets = config['target_cols']
    seed = frame[frame['year'].isin([origin - 1, origin])].dropna(subset=targets).sort_values(keys + ['year'])
    history = {}
    for key, group in seed.groupby(keys, sort=False):
        key = key if isinstance(key, tuple) else (key,)
        years = group['year'].tolist()
        if origin not in years:
            continue
        if config['feature_level'] == 'overall' and origin - 1 not in years:
            continue
        history[key] = group[targets].to_numpy(dtype=float).tolist()
    return history


def recursive_forecast_from_origin(model, model_name, numeric_for_model, interaction_cols, frame, config, origin):
    targets = config['target_cols']
    keys = config['panel_keys']
    lag1_cols = [f'{c}_lag1' for c in targets]
    extra_cols = [f'{c}_lag2' if config['feature_level'] == 'overall' else f'{c}_roll2' for c in targets]
    history = build_origin_history(frame, config, origin)
    records = []
    for forecast_year in range(origin + 1, YEAR_REF + 1):
        feature_rows = []
        active_keys = list(history.keys())
        for key in active_keys:
            values = history[key]
            last = values[-1]
            previous = values[-2] if len(values) >= 2 else last
            row = dict(zip(keys, key))
            for index, target in enumerate(targets):
                row[f'{target}_lag1'] = last[index]
                row[extra_cols[index]] = previous[index] if config['feature_level'] == 'overall' else (last[index] + previous[index]) / 2
            row['time_trend'] = forecast_year / YEAR_REF
            row['is_covid_year'] = int(forecast_year in [5, 6])
            feature_rows.append(row)
        features = pd.DataFrame(feature_rows)
        if features.empty:
            break
        if model_name == 'Naive baseline':
            predicted = features[lag1_cols].to_numpy(dtype=float)
        else:
            model_input, _ = add_interaction_terms(features, keys[1], interaction_cols)
            predicted = predict_one_model(model, model_input, config, numeric_for_model)
        step = features[keys].copy()
        step['origin'] = origin
        step['forecast_year'] = forecast_year
        step['horizon'] = forecast_year - origin
        step['selected_model'] = model_name
        for index, target in enumerate(targets):
            step[f'predicted_{target}'] = predicted[:, index]
        records.append(step)
        for index, key in enumerate(active_keys):
            history[key] = [history[key][-1], predicted[index].tolist()]
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def attach_actuals(predictions, frame, config):
    targets = config['target_cols']
    actual_cols = config['panel_keys'] + ['year'] + targets
    if config.get('n_col') in frame.columns:
        actual_cols.append(config['n_col'])
    actual = frame[actual_cols].copy().rename(columns={'year': 'forecast_year', **{c: f'actual_{c}' for c in targets}, **({config['n_col']: 'sample_n'} if config.get('n_col') in frame.columns else {})})
    return predictions.merge(actual, on=config['panel_keys'] + ['forecast_year'], how='inner')

## 3. Run and save one task at a time

In [6]:
def run_recursive_backtest(frame, config):
    started = time.time()
    prepared, lag1_cols, extra_cols, numeric, interaction_cols = prepare_task(frame, config)
    evaluated_frames = []
    selection_frames = []
    for origin in BACKTEST_ORIGINS:
        print(f"{config['task']}: origin {origin}")
        model, model_name, numeric_for_model, selection = select_and_refit_at_origin(prepared, config, origin, lag1_cols, extra_cols, numeric, interaction_cols)
        predictions = recursive_forecast_from_origin(model, model_name, numeric_for_model, interaction_cols, frame, config, origin)
        evaluated = attach_actuals(predictions, frame, config)
        evaluated_frames.append(evaluated)
        selection_frames.append(selection)
    evaluated_all = pd.concat(evaluated_frames, ignore_index=True)
    selections = pd.concat(selection_frames, ignore_index=True)
    by_origin_rows = []
    for (origin, horizon), group in evaluated_all.groupby(['origin', 'horizon'], sort=True):
        by_origin_rows.append({'task': config['task'], 'origin': origin, 'horizon': horizon, 'forecast_year': origin + horizon, 'selected_model': group['selected_model'].iloc[0], **score_prediction_frame(group, config)})
    by_origin = pd.DataFrame(by_origin_rows)
    by_horizon_rows = []
    for horizon, group in evaluated_all.groupby('horizon', sort=True):
        by_horizon_rows.append({'task': config['task'], 'horizon': horizon, 'origins': group['origin'].nunique(), **score_prediction_frame(group, config)})
    by_horizon = pd.DataFrame(by_horizon_rows)
    stem = config['task']
    by_origin.to_csv(BACKTEST_DIR / f'{stem}_metrics_by_origin.csv', index=False)
    by_horizon.to_csv(BACKTEST_DIR / f'{stem}_metrics_by_horizon.csv', index=False)
    selections.to_csv(BACKTEST_DIR / f'{stem}_model_selection.csv', index=False)
    if config['feature_level'] == 'overall' or SAVE_DETAILED_ACTIVITY_PREDICTIONS:
        evaluated_all.to_csv(BACKTEST_DIR / f'{stem}_predictions.csv.gz', index=False, compression='gzip')
    print(f"Completed {stem} in {(time.time() - started) / 60:.1f} minutes")
    display(by_horizon)
    return by_origin, by_horizon, selections

## 4. Load the eight historical panels

In [7]:
age_overall = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_activity = pd.read_csv(AGE_DATA_DIR / 'q3_age_activity_participation_level_panel_complete.csv')
disability_overall = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_overall_all_years.csv')
disability_activity = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_MEMS7GR_all_years.csv')
disability_rates = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_days_months_all_years.csv')

for frame in [age_overall, age_activity, disability_overall, disability_activity, disability_rates]:
    frame['LA_2023'] = frame['LA_2023'].astype('Int64').astype(str)

assert age_activity['activity_suffix'].nunique() == 124
assert disability_activity['activity'].nunique() == 124
assert disability_rates['activity'].nunique() == 124

print('Age overall rows:', len(age_overall))
print('Age activity rows:', len(age_activity))
print('Disability overall rows:', len(disability_overall))
print('Disability activity rows:', len(disability_activity))
print('Disability rates rows:', len(disability_rates))

Age overall rows: 2048
Age activity rows: 253952
Disability overall rows: 4096
Disability activity rows: 507904
Disability rates rows: 507904


In [8]:
TASKS = {
    'age_overall_level': {'task': 'age_overall_level', 'panel_keys': ['LA_2023', 'age_group'], 'target_cols': ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate'], 'weight_col': 'weighted_n_overall_activity_level', 'n_col': 'n_overall_activity_level', 'feature_level': 'overall', 'is_composition': True},
    'disability_overall_level': {'task': 'disability_overall_level', 'panel_keys': ['LA_2023', 'disability_group'], 'target_cols': ['inactive_rate', 'fairly_active_rate', 'active_rate'], 'weight_col': 'weighted_n', 'n_col': 'n', 'feature_level': 'overall', 'is_composition': True},
    'age_activity_level': {'task': 'age_activity_level', 'panel_keys': ['LA_2023', 'age_group', 'activity_suffix'], 'target_cols': ['activity_inactive_rate', 'activity_fairly_active_rate', 'activity_active_rate'], 'weight_col': 'weighted_n_activity_level', 'n_col': 'n_activity_level', 'feature_level': 'activity', 'is_composition': True},
    'disability_activity_level': {'task': 'disability_activity_level', 'panel_keys': ['LA_2023', 'disability_group', 'activity'], 'target_cols': ['inactive_rate', 'fairly_active_rate', 'active_rate'], 'weight_col': 'weighted_n', 'n_col': 'n', 'feature_level': 'activity', 'is_composition': True},
    'age_months12': {'task': 'age_months12', 'panel_keys': ['LA_2023', 'age_group', 'activity_suffix'], 'target_cols': ['months12_rate'], 'weight_col': 'weighted_n_months12', 'n_col': 'n_months12', 'feature_level': 'activity', 'is_composition': False},
    'disability_months12': {'task': 'disability_months12', 'panel_keys': ['LA_2023', 'disability_group', 'activity'], 'target_cols': ['participation_MONTHS_12'], 'weight_col': 'weighted_n_MONTHS_12', 'n_col': 'n_MONTHS_12', 'feature_level': 'activity', 'is_composition': False},
    'age_days10p60gr': {'task': 'age_days10p60gr', 'panel_keys': ['LA_2023', 'age_group', 'activity_suffix'], 'target_cols': ['days10p60gr_rate'], 'weight_col': 'weighted_n_days10p60gr', 'n_col': 'n_days10p60gr', 'feature_level': 'activity', 'is_composition': False},
    'disability_days10p60gr': {'task': 'disability_days10p60gr', 'panel_keys': ['LA_2023', 'disability_group', 'activity'], 'target_cols': ['participation_DAYS10P60GR'], 'weight_col': 'weighted_n_DAYS10P60GR', 'n_col': 'n_DAYS10P60GR', 'feature_level': 'activity', 'is_composition': False}
}

task_results = {}

## 5. Age overall activity level

In [9]:
task_results['age_overall_level'] = run_recursive_backtest(age_overall, TASKS['age_overall_level'])

age_overall_level: origin 4
age_overall_level: origin 5
age_overall_level: origin 6
age_overall_level: origin 7
Completed age_overall_level in 0.3 minutes


,task,horizon,origins,observations,total_variation,overall_mae,overall_rmse,overall_inactive_rate_mae,overall_inactive_rate_rmse,overall_inactive_rate_r2,...,overall_active_rate_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,age_overall_level,1,4,1024,0.111413,0.074275,0.109055,0.084591,0.123885,0.515843,...,0.550186,1001,0.083572,23,0.306036,220,0.214844,0.217956,804,0.082259
1,age_overall_level,2,3,768,0.108496,0.072331,0.105017,0.083058,0.120307,0.518532,...,0.549044,752,0.081821,16,0.274933,164,0.213542,0.205133,604,0.082257
2,age_overall_level,3,2,512,0.109910,0.073273,0.108580,0.084181,0.125426,0.458307,...,0.514735,503,0.084849,9,0.284691,108,0.210938,0.208854,404,0.083460
3,age_overall_level,4,1,256,0.111467,0.074311,0.111472,0.088663,0.129435,0.398212,...,0.452782,252,0.087771,4,0.278082,51,0.199219,0.219253,205,0.084651


## 6. Disability overall activity level

In [10]:
task_results['disability_overall_level'] = run_recursive_backtest(disability_overall, TASKS['disability_overall_level'])

disability_overall_level: origin 4
disability_overall_level: origin 5
disability_overall_level: origin 6
disability_overall_level: origin 7
Completed disability_overall_level in 0.3 minutes


,task,horizon,origins,observations,total_variation,overall_mae,overall_rmse,inactive_rate_mae,inactive_rate_rmse,inactive_rate_r2,...,active_rate_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,disability_overall_level,1,4,2000,0.197224,0.131483,0.187213,0.151276,0.204763,0.105085,...,0.103397,1906,0.143375,83,0.529946,1302,0.651000,0.253089,698,0.093898
1,disability_overall_level,2,3,1496,0.190739,0.127160,0.181128,0.147938,0.200031,0.125375,...,0.119236,1433,0.140380,57,0.520493,953,0.637032,0.246714,543,0.093119
2,disability_overall_level,3,2,997,0.185284,0.123523,0.175406,0.143770,0.194663,0.136973,...,0.135790,962,0.138815,34,0.501050,620,0.621866,0.241712,377,0.092635
3,disability_overall_level,4,1,495,0.181029,0.120686,0.168379,0.140562,0.189585,0.138774,...,0.148098,477,0.130257,18,0.501303,306,0.618182,0.236579,189,0.091090


## 7. Age activity-specific level

In [11]:
task_results['age_activity_level'] = run_recursive_backtest(age_activity, TASKS['age_activity_level'])

age_activity_level: origin 4
age_activity_level: origin 5
age_activity_level: origin 6
age_activity_level: origin 7
Completed age_activity_level in 43.2 minutes


,task,horizon,origins,observations,total_variation,overall_mae,overall_rmse,activity_inactive_rate_mae,activity_inactive_rate_rmse,activity_inactive_rate_r2,...,activity_active_rate_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,age_activity_level,1,4,126870,0.007517,0.005011,0.022539,0.007019,0.028249,0.721221,...,0.631587,13895,0.040484,112816,0.000157,31043,0.244684,0.005516,95827,0.008162
1,age_activity_level,2,3,95232,0.008220,0.005480,0.024405,0.007787,0.030866,0.683238,...,0.595541,10620,0.043728,84453,0.000101,24205,0.254169,0.005894,71027,0.009008
2,age_activity_level,3,2,63488,0.008845,0.005897,0.025722,0.008495,0.032976,0.668535,...,0.627383,7532,0.043326,55797,0.000039,16466,0.259356,0.006489,47022,0.009663
3,age_activity_level,4,1,31744,0.009260,0.006173,0.027210,0.009047,0.035562,0.630586,...,0.638732,3829,0.042805,27862,0.000002,7861,0.247637,0.007238,23883,0.009921


## 8. Disability activity-specific level

In [12]:
task_results['disability_activity_level'] = run_recursive_backtest(disability_activity, TASKS['disability_activity_level'])

disability_activity_level: origin 4
disability_activity_level: origin 5
disability_activity_level: origin 6
disability_activity_level: origin 7
Completed disability_activity_level in 98.1 minutes


,task,horizon,origins,observations,total_variation,overall_mae,overall_rmse,inactive_rate_mae,inactive_rate_rmse,inactive_rate_r2,...,active_rate_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,disability_activity_level,1,4,251013,0.008115,0.005410,0.036973,0.007723,0.046082,0.351367,...,0.271685,15464,0.075922,232965,0.000215,174266,0.694251,0.008342,76747,0.007606
1,disability_activity_level,2,3,188303,0.008372,0.005581,0.036727,0.007986,0.045809,0.362072,...,0.284307,11876,0.077075,174587,0.000230,130776,0.694498,0.008345,57527,0.008433
2,disability_activity_level,3,2,125612,0.008713,0.005809,0.036899,0.008360,0.046177,0.370719,...,0.298806,8335,0.076697,116358,0.000155,86602,0.689440,0.008429,39010,0.009337
3,disability_activity_level,4,1,62992,0.008416,0.005610,0.034783,0.008007,0.043319,0.414877,...,0.342756,4159,0.071658,58568,0.000222,43160,0.685166,0.008138,19832,0.009015


## 9. Age past-12-month participation

In [13]:
task_results['age_months12'] = run_recursive_backtest(age_activity, TASKS['age_months12'])

age_months12: origin 4
age_months12: origin 5
age_months12: origin 6
age_months12: origin 7
Completed age_months12 in 23.4 minutes


,task,horizon,origins,observations,months12_rate_mae,months12_rate_rmse,months12_rate_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,age_months12,1,4,126870,0.016387,0.044172,0.816072,40917,0.042170,85794,0.004090,31043,0.244684,0.014278,95827,0.017066
1,age_months12,2,3,95232,0.017405,0.046383,0.791911,30277,0.045475,64796,0.004289,24205,0.254169,0.014511,71027,0.018385
2,age_months12,3,2,63488,0.017707,0.047545,0.792521,21256,0.044292,42073,0.004276,16466,0.259356,0.015776,47022,0.018376
3,age_months12,4,1,31744,0.018540,0.047908,0.798235,10945,0.045294,20746,0.004425,7861,0.247637,0.017323,23883,0.018938


## 10. Disability past-12-month participation

In [14]:
task_results['disability_months12'] = run_recursive_backtest(disability_rates, TASKS['disability_months12'])

disability_months12: origin 4
disability_months12: origin 5
disability_months12: origin 6
disability_months12: origin 7
Completed disability_months12 in 61.0 minutes


,task,horizon,origins,observations,participation_MONTHS_12_mae,participation_MONTHS_12_rmse,participation_MONTHS_12_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,disability_months12,1,4,251013,0.020912,0.064815,0.585260,45977,0.072011,202452,0.009308,174266,0.694251,0.023216,76747,0.015758
1,disability_months12,2,3,188303,0.021332,0.065877,0.552380,34068,0.075653,152395,0.009188,130776,0.694498,0.023168,57527,0.017216
2,disability_months12,3,2,125612,0.022677,0.066167,0.557354,23994,0.075270,100699,0.010145,86602,0.689440,0.024811,39010,0.017989
3,disability_months12,4,1,62992,0.022898,0.066479,0.556434,12179,0.076342,50548,0.010021,43160,0.685166,0.024955,19832,0.018447


## 11. Age regular participation

In [15]:
task_results['age_days10p60gr'] = run_recursive_backtest(age_activity, TASKS['age_days10p60gr'])

age_days10p60gr: origin 4
age_days10p60gr: origin 5
age_days10p60gr: origin 6
age_days10p60gr: origin 7
Completed age_days10p60gr in 25.3 minutes


,task,horizon,origins,observations,days10p60gr_rate_mae,days10p60gr_rate_rmse,days10p60gr_rate_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,age_days10p60gr,1,4,126870,0.008499,0.031401,0.744597,20550,0.040847,106161,0.002237,31043,0.244684,0.008823,95827,0.008395
1,age_days10p60gr,2,3,95232,0.009204,0.033565,0.721082,15803,0.044464,79270,0.002175,24205,0.254169,0.009056,71027,0.009254
2,age_days10p60gr,3,2,63488,0.009613,0.033266,0.746839,11220,0.042010,52109,0.002637,16466,0.259356,0.009657,47022,0.009598
3,age_days10p60gr,4,1,31744,0.009326,0.035109,0.728864,5717,0.043732,25974,0.001753,7861,0.247637,0.010214,23883,0.009036


## 12. Disability regular participation

In [16]:
task_results['disability_days10p60gr'] = run_recursive_backtest(disability_rates, TASKS['disability_days10p60gr'])

disability_days10p60gr: origin 4
disability_days10p60gr: origin 5
disability_days10p60gr: origin 6
disability_days10p60gr: origin 7
Completed disability_days10p60gr in 401.5 minutes


,task,horizon,origins,observations,participation_DAYS10P60GR_mae,participation_DAYS10P60GR_rmse,participation_DAYS10P60GR_r2,positive_cell_observations,positive_cell_mae,zero_cell_observations,zero_cell_mae,small_n_observations,small_n_rate,small_n_mean_error,large_n_observations,large_n_mean_error
0,disability_days10p60gr,1,4,251013,0.009186,0.043677,0.510071,23355,0.069751,225074,0.002902,174266,0.694251,0.010101,76747,0.007141
1,disability_days10p60gr,2,3,188303,0.009611,0.043859,0.508989,17874,0.071125,168589,0.003089,130776,0.694498,0.010329,57527,0.008003
2,disability_days10p60gr,3,2,125612,0.009146,0.041283,0.574945,12577,0.068158,112116,0.002526,86602,0.689440,0.009664,39010,0.008007
3,disability_days10p60gr,4,1,62992,0.009007,0.040482,0.577038,6312,0.066752,56415,0.002546,43160,0.685166,0.009518,19832,0.007902


## 13. Combined summaries and checks

In [17]:
metric_files = sorted(BACKTEST_DIR.glob('*_metrics_by_horizon.csv'))
origin_files = sorted(BACKTEST_DIR.glob('*_metrics_by_origin.csv'))
selection_files = sorted(BACKTEST_DIR.glob('*_model_selection.csv'))

combined_horizon = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
combined_origin = pd.concat([pd.read_csv(path) for path in origin_files], ignore_index=True)
combined_selection = pd.concat([pd.read_csv(path) for path in selection_files], ignore_index=True)

combined_horizon.to_csv(BACKTEST_DIR / 'all_tasks_metrics_by_horizon.csv', index=False)
combined_origin.to_csv(BACKTEST_DIR / 'all_tasks_metrics_by_origin.csv', index=False)
combined_selection.to_csv(BACKTEST_DIR / 'all_tasks_model_selection.csv', index=False)

expected_tasks = set(TASKS)
completed_tasks = set(combined_horizon['task'])
assert completed_tasks == expected_tasks, f'Missing tasks: {sorted(expected_tasks - completed_tasks)}'
assert set(combined_horizon['horizon']) == {1, 2, 3, 4}
assert combined_horizon.groupby('task')['horizon'].nunique().eq(4).all()
assert combined_selection.groupby(['task', 'origin'])['selected'].sum().eq(1).all()

print('Completed tasks:', sorted(completed_tasks))
print('Files saved to:', BACKTEST_DIR)
display(combined_horizon.sort_values(['task', 'horizon']))

Completed tasks: ['age_activity_level', 'age_days10p60gr', 'age_months12', 'age_overall_level', 'disability_activity_level', 'disability_days10p60gr', 'disability_months12', 'disability_overall_level']
Files saved to: C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs\recursive_backtest


,task,horizon,origins,observations,total_variation,overall_mae,overall_rmse,activity_inactive_rate_mae,activity_inactive_rate_rmse,activity_inactive_rate_r2,...,fairly_active_rate_r2,active_rate_mae,active_rate_rmse,active_rate_r2,participation_DAYS10P60GR_mae,participation_DAYS10P60GR_rmse,participation_DAYS10P60GR_r2,participation_MONTHS_12_mae,participation_MONTHS_12_rmse,participation_MONTHS_12_r2
0,age_activity_level,1,4,126870,0.007517,0.005011,0.022539,0.007019,0.028249,0.721221,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,age_activity_level,2,3,95232,0.008220,0.005480,0.024405,0.007787,0.030866,0.683238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,age_activity_level,3,2,63488,0.008845,0.005897,0.025722,0.008495,0.032976,0.668535,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,age_activity_level,4,1,31744,0.009260,0.006173,0.027210,0.009047,0.035562,0.630586,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,age_days10p60gr,1,4,126870,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,age_days10p60gr,2,3,95232,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,age_days10p60gr,3,2,63488,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,age_days10p60gr,4,1,31744,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,age_months12,1,4,126870,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,age_months12,2,3,95232,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 14. Compact reporting table

In [18]:
report_rows = []
for _, row in combined_horizon.iterrows():
    task = row['task']
    if task in ['age_overall_level', 'disability_overall_level', 'age_activity_level', 'disability_activity_level']:
        primary_metric = 'total_variation'
        primary_value = row['total_variation']
    else:
        target = TASKS[task]['target_cols'][0]
        primary_metric = f'{target}_mae'
        primary_value = row[primary_metric]
    report_rows.append({'task': task, 'horizon': int(row['horizon']), 'origins': int(row['origins']), 'observations': int(row['observations']), 'primary_metric': primary_metric, 'primary_value': primary_value, 'positive_cell_mae': row.get('positive_cell_mae', np.nan), 'zero_cell_mae': row.get('zero_cell_mae', np.nan), 'small_n_rate': row.get('small_n_rate', np.nan), 'small_n_mean_error': row.get('small_n_mean_error', np.nan), 'large_n_mean_error': row.get('large_n_mean_error', np.nan)})

reporting_table = pd.DataFrame(report_rows)
reporting_table.to_csv(BACKTEST_DIR / 'recursive_backtest_reporting_table.csv', index=False)
display(reporting_table.round(4))

,task,horizon,origins,observations,primary_metric,primary_value,positive_cell_mae,zero_cell_mae,small_n_rate,small_n_mean_error,large_n_mean_error
0,age_activity_level,1,4,126870,total_variation,0.0075,0.0405,0.0002,0.2447,0.0055,0.0082
1,age_activity_level,2,3,95232,total_variation,0.0082,0.0437,0.0001,0.2542,0.0059,0.0090
2,age_activity_level,3,2,63488,total_variation,0.0088,0.0433,0.0000,0.2594,0.0065,0.0097
3,age_activity_level,4,1,31744,total_variation,0.0093,0.0428,0.0000,0.2476,0.0072,0.0099
4,age_days10p60gr,1,4,126870,days10p60gr_rate_mae,0.0085,0.0408,0.0022,0.2447,0.0088,0.0084
5,age_days10p60gr,2,3,95232,days10p60gr_rate_mae,0.0092,0.0445,0.0022,0.2542,0.0091,0.0093
6,age_days10p60gr,3,2,63488,days10p60gr_rate_mae,0.0096,0.0420,0.0026,0.2594,0.0097,0.0096
7,age_days10p60gr,4,1,31744,days10p60gr_rate_mae,0.0093,0.0437,0.0018,0.2476,0.0102,0.0090
8,age_months12,1,4,126870,months12_rate_mae,0.0164,0.0422,0.0041,0.2447,0.0143,0.0171
9,age_months12,2,3,95232,months12_rate_mae,0.0174,0.0455,0.0043,0.2542,0.0145,0.0184


## 15. Recursive Naive benchmark comparison

In [19]:
TASK_FRAMES = {
    'age_overall_level': age_overall,
    'disability_overall_level': disability_overall,
    'age_activity_level': age_activity,
    'disability_activity_level': disability_activity,
    'age_months12': age_activity,
    'disability_months12': disability_rates,
    'age_days10p60gr': age_activity,
    'disability_days10p60gr': disability_rates
}


def primary_metric_for_task(task_name):
    config = TASKS[task_name]
    if config['is_composition']:
        return 'total_variation'
    return f"{config['target_cols'][0]}_mae"


def run_naive_recursive_backtest(frame, config):
    evaluated_frames = []

    for origin in BACKTEST_ORIGINS:
        predictions = recursive_forecast_from_origin(
            model=None,
            model_name='Naive baseline',
            numeric_for_model=[],
            interaction_cols=[],
            frame=frame,
            config=config,
            origin=origin
        )

        evaluated = attach_actuals(
            predictions,
            frame,
            config
        )
        evaluated_frames.append(evaluated)

    evaluated_all = pd.concat(
        evaluated_frames,
        ignore_index=True
    )

    by_origin_rows = []
    for (origin, horizon), group in evaluated_all.groupby(
        ['origin', 'horizon'],
        sort=True
    ):
        by_origin_rows.append({
            'task': config['task'],
            'origin': int(origin),
            'horizon': int(horizon),
            'forecast_year': int(origin + horizon),
            'model': 'Naive baseline',
            **score_prediction_frame(group, config)
        })

    by_horizon_rows = []
    for horizon, group in evaluated_all.groupby(
        'horizon',
        sort=True
    ):
        by_horizon_rows.append({
            'task': config['task'],
            'horizon': int(horizon),
            'origins': int(group['origin'].nunique()),
            'model': 'Naive baseline',
            **score_prediction_frame(group, config)
        })

    return (
        pd.DataFrame(by_origin_rows),
        pd.DataFrame(by_horizon_rows)
    )


naive_origin_frames = []
naive_horizon_frames = []

for task_name, config in TASKS.items():
    print(
        f'Running recursive Naive benchmark for {task_name}'
    )

    naive_by_origin, naive_by_horizon = (
        run_naive_recursive_backtest(
            TASK_FRAMES[task_name],
            config
        )
    )

    naive_origin_frames.append(naive_by_origin)
    naive_horizon_frames.append(naive_by_horizon)


all_naive_by_origin = pd.concat(
    naive_origin_frames,
    ignore_index=True
)

all_naive_by_horizon = pd.concat(
    naive_horizon_frames,
    ignore_index=True
)

all_naive_by_origin.to_csv(
    BACKTEST_DIR / 'all_tasks_naive_metrics_by_origin.csv',
    index=False
)

all_naive_by_horizon.to_csv(
    BACKTEST_DIR / 'all_tasks_naive_metrics_by_horizon.csv',
    index=False
)


selected_by_origin = pd.read_csv(
    BACKTEST_DIR / 'all_tasks_metrics_by_origin.csv'
)

selected_by_horizon = pd.read_csv(
    BACKTEST_DIR / 'all_tasks_metrics_by_horizon.csv'
)


origin_comparison_rows = []

for _, selected_row in selected_by_origin.iterrows():
    task_name = selected_row['task']
    primary_metric = primary_metric_for_task(task_name)

    naive_row = all_naive_by_origin[
        (all_naive_by_origin['task'] == task_name)
        & (
            all_naive_by_origin['origin']
            == selected_row['origin']
        )
        & (
            all_naive_by_origin['horizon']
            == selected_row['horizon']
        )
    ].iloc[0]

    selected_value = float(
        selected_row[primary_metric]
    )

    naive_value = float(
        naive_row[primary_metric]
    )

    absolute_improvement = (
        naive_value - selected_value
    )

    percentage_improvement = (
        100 * absolute_improvement / naive_value
        if np.isfinite(naive_value) and naive_value > 0
        else np.nan
    )

    origin_comparison_rows.append({
        'task': task_name,
        'origin': int(selected_row['origin']),
        'horizon': int(selected_row['horizon']),
        'forecast_year': int(
            selected_row['forecast_year']
        ),
        'selected_model': selected_row['selected_model'],
        'primary_metric': primary_metric,
        'selected_primary_value': selected_value,
        'naive_primary_value': naive_value,
        'absolute_improvement_over_naive': (
            absolute_improvement
        ),
        'percentage_improvement_over_naive': (
            percentage_improvement
        ),
        'selected_model_better_than_naive': bool(
            selected_value < naive_value
        )
    })


horizon_comparison_rows = []

for _, selected_row in selected_by_horizon.iterrows():
    task_name = selected_row['task']
    primary_metric = primary_metric_for_task(task_name)

    naive_row = all_naive_by_horizon[
        (all_naive_by_horizon['task'] == task_name)
        & (
            all_naive_by_horizon['horizon']
            == selected_row['horizon']
        )
    ].iloc[0]

    selected_value = float(
        selected_row[primary_metric]
    )

    naive_value = float(
        naive_row[primary_metric]
    )

    absolute_improvement = (
        naive_value - selected_value
    )

    percentage_improvement = (
        100 * absolute_improvement / naive_value
        if np.isfinite(naive_value) and naive_value > 0
        else np.nan
    )

    horizon_comparison_rows.append({
        'task': task_name,
        'horizon': int(selected_row['horizon']),
        'origins': int(selected_row['origins']),
        'observations': int(
            selected_row['observations']
        ),
        'primary_metric': primary_metric,
        'selected_primary_value': selected_value,
        'naive_primary_value': naive_value,
        'absolute_improvement_over_naive': (
            absolute_improvement
        ),
        'percentage_improvement_over_naive': (
            percentage_improvement
        ),
        'selected_model_better_than_naive': bool(
            selected_value < naive_value
        )
    })


selected_vs_naive_by_origin = pd.DataFrame(
    origin_comparison_rows
)

selected_vs_naive_by_horizon = pd.DataFrame(
    horizon_comparison_rows
)

selected_vs_naive_by_origin.to_csv(
    BACKTEST_DIR / 'selected_vs_naive_by_origin.csv',
    index=False
)

selected_vs_naive_by_horizon.to_csv(
    BACKTEST_DIR / 'selected_vs_naive_by_horizon.csv',
    index=False
)


naive_summary = (
    selected_vs_naive_by_horizon
    .groupby('task', as_index=False)
    .agg(
        horizons_better_than_naive=(
            'selected_model_better_than_naive',
            'sum'
        ),
        mean_percentage_improvement=(
            'percentage_improvement_over_naive',
            'mean'
        ),
        minimum_percentage_improvement=(
            'percentage_improvement_over_naive',
            'min'
        ),
        maximum_percentage_improvement=(
            'percentage_improvement_over_naive',
            'max'
        )
    )
)

naive_summary.to_csv(
    BACKTEST_DIR / 'selected_vs_naive_task_summary.csv',
    index=False
)


assert len(all_naive_by_horizon) == len(TASKS) * 4

assert (
    len(selected_vs_naive_by_horizon)
    == len(TASKS) * 4
)

print(
    'Naive recursive comparison completed and saved to:',
    BACKTEST_DIR
)

display(
    selected_vs_naive_by_horizon.round(4)
)

display(
    naive_summary.round(3)
)

Running recursive Naive benchmark for age_overall_level
Running recursive Naive benchmark for disability_overall_level
Running recursive Naive benchmark for age_activity_level
Running recursive Naive benchmark for disability_activity_level
Running recursive Naive benchmark for age_months12
Running recursive Naive benchmark for disability_months12
Running recursive Naive benchmark for age_days10p60gr
Running recursive Naive benchmark for disability_days10p60gr
Naive recursive comparison completed and saved to: C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs\recursive_backtest


,task,horizon,origins,observations,primary_metric,selected_primary_value,naive_primary_value,absolute_improvement_over_naive,percentage_improvement_over_naive,selected_model_better_than_naive
0,age_activity_level,1,4,126870,total_variation,0.0075,0.0088,0.0013,14.5465,True
1,age_activity_level,2,3,95232,total_variation,0.0082,0.0092,0.0009,10.2410,True
2,age_activity_level,3,2,63488,total_variation,0.0088,0.0093,0.0005,4.9880,True
3,age_activity_level,4,1,31744,total_variation,0.0093,0.0094,0.0002,1.9518,True
4,age_days10p60gr,1,4,126870,days10p60gr_rate_mae,0.0085,0.0086,0.0001,0.8657,True
5,age_days10p60gr,2,3,95232,days10p60gr_rate_mae,0.0092,0.0090,-0.0002,-2.2294,False
6,age_days10p60gr,3,2,63488,days10p60gr_rate_mae,0.0096,0.0091,-0.0005,-5.2496,False
7,age_days10p60gr,4,1,31744,days10p60gr_rate_mae,0.0093,0.0093,0.0000,0.0000,True
8,age_months12,1,4,126870,months12_rate_mae,0.0164,0.0164,0.0000,0.0000,True
9,age_months12,2,3,95232,months12_rate_mae,0.0174,0.0174,0.0000,0.0000,True


,task,horizons_better_than_naive,mean_percentage_improvement,minimum_percentage_improvement,maximum_percentage_improvement
0,age_activity_level,4,7.932,1.952,14.547
1,age_days10p60gr,2,-1.653,-5.250,0.866
2,age_months12,4,0.000,0.000,0.000
3,age_overall_level,4,24.686,24.092,25.556
4,disability_activity_level,4,23.795,21.649,27.044
5,disability_days10p60gr,4,11.867,9.774,13.473
6,disability_months12,3,2.132,-0.184,4.676
7,disability_overall_level,4,26.498,24.927,28.573


## 16. No-look-ahead COVID sensitivity

In [20]:
def recursive_forecast_without_future_covid(model, model_name, numeric_for_model, interaction_cols, frame, config, origin):
    targets = config['target_cols']
    keys = config['panel_keys']
    lag1_cols = [f'{c}_lag1' for c in targets]
    extra_cols = [f'{c}_lag2' if config['feature_level'] == 'overall' else f'{c}_roll2' for c in targets]
    history = build_origin_history(frame, config, origin)
    records = []
    for forecast_year in range(origin + 1, YEAR_REF + 1):
        feature_rows = []
        active_keys = list(history.keys())
        for key in active_keys:
            values = history[key]
            last = values[-1]
            previous = values[-2] if len(values) >= 2 else last
            row = dict(zip(keys, key))
            for index, target in enumerate(targets):
                row[f'{target}_lag1'] = last[index]
                row[extra_cols[index]] = previous[index] if config['feature_level'] == 'overall' else (last[index] + previous[index]) / 2
            row['time_trend'] = forecast_year / YEAR_REF
            row['is_covid_year'] = 0
            feature_rows.append(row)
        features = pd.DataFrame(feature_rows)
        if features.empty:
            break
        if model_name == 'Naive baseline':
            predicted = features[lag1_cols].to_numpy(dtype=float)
        else:
            model_input, _ = add_interaction_terms(features, keys[1], interaction_cols)
            predicted = predict_one_model(model, model_input, config, numeric_for_model)
        step = features[keys].copy()
        step['origin'] = origin
        step['forecast_year'] = forecast_year
        step['horizon'] = forecast_year - origin
        step['selected_model'] = model_name
        for index, target in enumerate(targets):
            step[f'predicted_{target}'] = predicted[:, index]
        records.append(step)
        for index, key in enumerate(active_keys):
            history[key] = [history[key][-1], predicted[index].tolist()]
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def selected_configuration(selection_table, task_name, origin):
    selected_flag = selection_table['selected'].astype(str).str.strip().str.lower().isin(['true', '1'])
    selected_rows = selection_table[selected_flag & (selection_table['task'] == task_name) & (selection_table['origin'].astype(int) == int(origin))]
    if len(selected_rows) != 1:
        raise ValueError(f'Expected one selected model for {task_name}, origin {origin}; found {len(selected_rows)}')
    selected_row = selected_rows.iloc[0]
    return selected_row['model'], json.loads(selected_row['parameters'])


def refit_selected_model_for_origin(frame, config, origin, model_name, parameters):
    prepared, lag1_cols, extra_cols, numeric, interaction_cols = prepare_task(frame, config)
    refit = prepared[prepared['year'] <= origin].dropna(subset=config['target_cols'] + [config['weight_col']]).copy()
    refit = refit[pd.to_numeric(refit[config['weight_col']], errors='coerce') > 0].copy()
    if refit.empty:
        raise ValueError(f'{config["task"]}, origin {origin}: empty refit data')
    if model_name == 'Naive baseline':
        model = None
        numeric_for_model = numeric
    else:
        model, numeric_for_model = fit_one_model(model_name, parameters, refit, config, numeric, interaction_cols)
    return model, numeric_for_model, interaction_cols


selection_table = pd.read_csv(BACKTEST_DIR / 'all_tasks_model_selection.csv')
original_origin_metrics = pd.read_csv(BACKTEST_DIR / 'all_tasks_metrics_by_origin.csv')
original_reporting_table = pd.read_csv(BACKTEST_DIR / 'recursive_backtest_reporting_table.csv')

no_future_covid_metric_rows = []
for task_name, config in TASKS.items():
    frame = TASK_FRAMES[task_name]
    for origin in [4, 5]:
        model_name, parameters = selected_configuration(selection_table, task_name, origin)
        print(f'Refitting {task_name}, origin {origin}, selected model: {model_name}')
        model, numeric_for_model, interaction_cols = refit_selected_model_for_origin(frame, config, origin, model_name, parameters)
        predictions = recursive_forecast_without_future_covid(model, model_name, numeric_for_model, interaction_cols, frame, config, origin)
        evaluated = attach_actuals(predictions, frame, config)
        for horizon, group in evaluated.groupby('horizon', sort=True):
            no_future_covid_metric_rows.append({'task': task_name, 'origin': origin, 'horizon': horizon, 'forecast_year': origin + horizon, 'selected_model': model_name, **score_prediction_frame(group, config)})

no_future_covid_by_origin = pd.DataFrame(no_future_covid_metric_rows)
no_future_covid_by_origin.to_csv(BACKTEST_DIR / 'covid_no_lookahead_metrics_by_origin.csv', index=False)

covid_comparison_rows = []
for _, sensitivity_row in no_future_covid_by_origin.iterrows():
    task_name = sensitivity_row['task']
    primary_metric = primary_metric_for_task(task_name)
    original_row = original_origin_metrics[(original_origin_metrics['task'] == task_name) & (original_origin_metrics['origin'] == sensitivity_row['origin']) & (original_origin_metrics['horizon'] == sensitivity_row['horizon'])].iloc[0]
    original_value = float(original_row[primary_metric])
    no_lookahead_value = float(sensitivity_row[primary_metric])
    percentage_change = 100 * (no_lookahead_value - original_value) / original_value if np.isfinite(original_value) and original_value > 0 else np.nan
    covid_comparison_rows.append({'task': task_name, 'origin': int(sensitivity_row['origin']), 'horizon': int(sensitivity_row['horizon']), 'forecast_year': int(sensitivity_row['forecast_year']), 'selected_model': sensitivity_row['selected_model'], 'primary_metric': primary_metric, 'original_known_covid_value': original_value, 'no_future_covid_value': no_lookahead_value, 'absolute_change': no_lookahead_value - original_value, 'percentage_change': percentage_change})

covid_sensitivity_comparison = pd.DataFrame(covid_comparison_rows)
covid_sensitivity_comparison.to_csv(BACKTEST_DIR / 'covid_sensitivity_comparison_by_origin.csv', index=False)

unaffected_origin_metrics = original_origin_metrics[original_origin_metrics['origin'].isin([6, 7])].copy()
corrected_origin_metrics = pd.concat([no_future_covid_by_origin, unaffected_origin_metrics], ignore_index=True, sort=False)

corrected_reporting_rows = []
for (task_name, horizon), group in corrected_origin_metrics.groupby(['task', 'horizon'], sort=True):
    primary_metric = primary_metric_for_task(task_name)
    observation_weights = group['observations'].to_numpy(dtype=float)
    primary_values = group[primary_metric].to_numpy(dtype=float)
    positive_counts = group['positive_cell_observations'].to_numpy(dtype=float)
    positive_values = group['positive_cell_mae'].to_numpy(dtype=float)
    zero_counts = group['zero_cell_observations'].to_numpy(dtype=float)
    zero_values = group['zero_cell_mae'].to_numpy(dtype=float)
    small_counts = group['small_n_observations'].to_numpy(dtype=float)
    small_values = group['small_n_mean_error'].to_numpy(dtype=float)
    large_counts = group['large_n_observations'].to_numpy(dtype=float)
    large_values = group['large_n_mean_error'].to_numpy(dtype=float)
    corrected_reporting_rows.append({'task': task_name, 'horizon': int(horizon), 'origins': int(group['origin'].nunique()), 'observations': int(observation_weights.sum()), 'primary_metric': primary_metric, 'primary_value': np.average(primary_values, weights=observation_weights), 'positive_cell_mae': np.average(positive_values, weights=positive_counts) if positive_counts.sum() > 0 else np.nan, 'zero_cell_mae': np.average(zero_values, weights=zero_counts) if zero_counts.sum() > 0 else np.nan, 'small_n_rate': small_counts.sum() / observation_weights.sum() if observation_weights.sum() > 0 else np.nan, 'small_n_mean_error': np.average(small_values, weights=small_counts) if small_counts.sum() > 0 else np.nan, 'large_n_mean_error': np.average(large_values, weights=large_counts) if large_counts.sum() > 0 else np.nan})

covid_corrected_reporting_table = pd.DataFrame(corrected_reporting_rows)
covid_corrected_reporting_table.to_csv(BACKTEST_DIR / 'recursive_backtest_reporting_table_no_covid_lookahead.csv', index=False)

reporting_comparison = original_reporting_table.merge(covid_corrected_reporting_table, on=['task', 'horizon'], suffixes=('_original', '_no_covid_lookahead'))
reporting_comparison['primary_value_change'] = reporting_comparison['primary_value_no_covid_lookahead'] - reporting_comparison['primary_value_original']
reporting_comparison['primary_value_percentage_change'] = 100 * reporting_comparison['primary_value_change'] / reporting_comparison['primary_value_original']
reporting_comparison.to_csv(BACKTEST_DIR / 'recursive_backtest_covid_sensitivity_reporting_comparison.csv', index=False)

assert len(no_future_covid_by_origin) == 56
assert len(covid_corrected_reporting_table) == len(TASKS) * 4

print('No-look-ahead COVID sensitivity completed and saved to:', BACKTEST_DIR)
display(covid_sensitivity_comparison.round(4))
display(reporting_comparison[['task', 'horizon', 'primary_metric_original', 'primary_value_original', 'primary_value_no_covid_lookahead', 'primary_value_change', 'primary_value_percentage_change']].round(4))

Refitting age_overall_level, origin 4, selected model: Gradient Boosting
Refitting age_overall_level, origin 5, selected model: Gradient Boosting
Refitting disability_overall_level, origin 4, selected model: Gradient Boosting
Refitting disability_overall_level, origin 5, selected model: Gradient Boosting
Refitting age_activity_level, origin 4, selected model: Random Forest
Refitting age_activity_level, origin 5, selected model: Gradient Boosting
Refitting disability_activity_level, origin 4, selected model: Gradient Boosting
Refitting disability_activity_level, origin 5, selected model: Gradient Boosting
Refitting age_months12, origin 4, selected model: Naive baseline
Refitting age_months12, origin 5, selected model: Naive baseline
Refitting disability_months12, origin 4, selected model: Ridge Regression
Refitting disability_months12, origin 5, selected model: Ridge Regression
Refitting age_days10p60gr, origin 4, selected model: Naive baseline
Refitting age_days10p60gr, origin 5, selec

,task,origin,horizon,forecast_year,selected_model,primary_metric,original_known_covid_value,no_future_covid_value,absolute_change,percentage_change
0,age_overall_level,4,1,5,Gradient Boosting,total_variation,0.1230,0.1230,0.0000,0.0000
1,age_overall_level,4,2,6,Gradient Boosting,total_variation,0.1127,0.1127,0.0000,0.0000
2,age_overall_level,4,3,7,Gradient Boosting,total_variation,0.1088,0.1088,0.0000,0.0000
3,age_overall_level,4,4,8,Gradient Boosting,total_variation,0.1115,0.1115,0.0000,0.0000
4,age_overall_level,5,1,6,Gradient Boosting,total_variation,0.1090,0.1099,0.0009,0.8296
5,age_overall_level,5,2,7,Gradient Boosting,total_variation,0.1056,0.1055,-0.0001,-0.1209
6,age_overall_level,5,3,8,Gradient Boosting,total_variation,0.1110,0.1108,-0.0002,-0.1387
7,disability_overall_level,4,1,5,Gradient Boosting,total_variation,0.2165,0.2165,0.0000,0.0000
8,disability_overall_level,4,2,6,Gradient Boosting,total_variation,0.2042,0.2042,0.0000,0.0000
9,disability_overall_level,4,3,7,Gradient Boosting,total_variation,0.1893,0.1893,0.0000,0.0000


,task,horizon,primary_metric_original,primary_value_original,primary_value_no_covid_lookahead,primary_value_change,primary_value_percentage_change
0,age_activity_level,1,total_variation,0.0075,0.0075,0.0000,0.0099
1,age_activity_level,2,total_variation,0.0082,0.0082,0.0000,0.0052
2,age_activity_level,3,total_variation,0.0088,0.0088,-0.0000,-0.0030
3,age_activity_level,4,total_variation,0.0093,0.0093,0.0000,0.0000
4,age_days10p60gr,1,days10p60gr_rate_mae,0.0085,0.0085,0.0000,0.1310
5,age_days10p60gr,2,days10p60gr_rate_mae,0.0092,0.0092,-0.0000,-0.2092
6,age_days10p60gr,3,days10p60gr_rate_mae,0.0096,0.0096,-0.0001,-0.6282
7,age_days10p60gr,4,days10p60gr_rate_mae,0.0093,0.0093,0.0000,0.0000
8,age_months12,1,months12_rate_mae,0.0164,0.0164,-0.0000,-0.0042
9,age_months12,2,months12_rate_mae,0.0174,0.0174,-0.0000,-0.0055
